In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass
import math

In [9]:
@dataclass
class GQAConfig:
    hidden_dim: int
    q_heads: int
    kv_heads: int
    dropout: float

In [13]:
class GroupQueryAttention(nn.Module):
    def __init__(self, config: GQAConfig):
        super().__init__()
        self.config = config
        self.num_heads = config.q_heads
        self.kv_heads = config.kv_heads
        self.head_dim = config.hidden_dim // config.q_heads
        self.q_proj = nn.Linear(config.hidden_dim, self.num_heads * self.head_dim)
        self.k_proj = nn.Linear(config.hidden_dim, self.kv_heads * self.head_dim)
        self.v_proj = nn.Linear(config.hidden_dim, self.kv_heads * self.head_dim)
        self.o_proj = nn.Linear(config.hidden_dim, config.hidden_dim)
        self.dropout = nn.Dropout(config.dropout)
    
    def forward(self, x, attention_mask=None):
        # x (batch, seq_len, dim)
        q = self.q_proj(x)  #  (batch, seq_len, num_heads * head_dim)
        k = self.k_proj(x)
        v = self.v_proj(x)

        batch, seq, _ = x.size()
        q = q.view(batch, seq, self.num_heads, self.head_dim).transpose(1, 2)  # (batch, num_heads, seq, dim)
        k = k.view(batch, seq, self.kv_heads, self.head_dim).transpose(1, 2)   # (batch, kv_heads, seq, dim)
        v = v.view(batch, seq, self.kv_heads, self.head_dim).transpose(1, 2)

        k = k.repeat_interleave(self.num_heads // self.kv_heads, dim=1)
        v = v.repeat_interleave(self.num_heads // self.kv_heads, dim=1)

        attention_weight = q @ k.transpose(-1, -2) / math.sqrt(self.head_dim) #(batch, num_heads, seq, seq)
        if attention_mask is not None:
            attention_weight = attention_weight.masked_fill(
                attention_mask == 0, 
                float('-inf')
            )
        attention_weight = F.softmax(attention_weight, dim=-1)
        attention_weight = self.dropout(attention_weight)
        output = attention_weight @ v # (batch, num_heads, seq, dim)
        output = output.transpose(1, 2).contiguous()
        output = output.view(batch, seq, -1)
        return self.o_proj(output)

In [14]:
x = torch.rand(2, 4, 256)
config = GQAConfig(hidden_dim=256, q_heads=8, kv_heads=4, dropout=0.1)
GQA = GroupQueryAttention(config)
attention_mask = torch.tensor([
    [1, 1, 1, 0],
    [1, 1, 0, 0]
]).unsqueeze(1).unsqueeze(2).expand(2, 8, 4, 4)
output = GQA(x, attention_mask=attention_mask)
output

tensor([[[ 0.0419, -0.0849, -0.3936,  ...,  0.0072, -0.1332, -0.1909],
         [ 0.0575,  0.0420, -0.3922,  ...,  0.0144, -0.0299, -0.2484],
         [-0.0061,  0.0170, -0.4105,  ...,  0.0247,  0.0282, -0.2750],
         [ 0.0276, -0.0026, -0.4294,  ...,  0.0146, -0.0186, -0.2862]],

        [[-0.0136, -0.1354, -0.4480,  ...,  0.0643, -0.0814, -0.1976],
         [ 0.0210, -0.1915, -0.3783,  ..., -0.0050, -0.1426, -0.1374],
         [ 0.0110, -0.2029, -0.4448,  ...,  0.0521, -0.1150, -0.1689],
         [ 0.0335, -0.1530, -0.4072,  ...,  0.0727, -0.1134, -0.1890]]],
       grad_fn=<ViewBackward0>)

In [ ]:
@dataclass
class DeepseekConfig:
    hidden_size: int
    num_heads: int
    max_position_embeddings: int
    rope_theta: float
    dropout: float

    q_lora_rank: int   # q 降维后的维度
    qk_rope_head_dim: int  # 对应 query 和 key 进行 rope 的维度
    kv_lora_rank: int
    v_head_dim: int
    qk_nope_head_dim: int  # 对应 query 和 key 不进行 rope 的维度
    bias: bool

In [ ]:

class MLA(nn.Module):
    def __init__(self, config,):
        super().__init__()
        
        self.attention_dropout = config.attention_dropout
        self.hidden_size = config.hidden_size
        self.num_heads = config.num_heads

        self.max_postion_embeddings = config.max_position_embeddings
        self.rope_theta = config.rope_theta

        # 对应 query 压缩的向量， 在 deepseek v3 中， hidden_size 7168
        # 但是压缩后的 kv d_c= 512，压缩比例 1/14
        # q 的压缩为 1536 压缩比例 1/4.7
        # rope 部分是 64

        self.q_lora_rank = config.q_lora_rank
        # 对应 query 和 key 进行 rope 的维度
        self.qk_rope_head_dim = config.qk_rope_head_dim
        # 对应 value 压缩的向量
        self.kv_lora_rank = config.kv_lora_rank
        
        # 对应 每一个 Head 的维度大小
        self.v_head_dim = config.v_head_dim

        self.qk_nope_head_dim = config.qk_nope_head_dim
        self.q_head_dim = config.qk_nope_head_dim + config.qk_rope_head_dim

        self.q_down_proj = nn.Linear(
            self.hidden_size,
            self.q_lora_rank,
            bias=config.attention_bias,
        )
        self.q_down_layernorm = DeepseekV2RMSNorm(self.q_lora_rank)

        self.q_up_proj = nn.Linear(
            self.q_lora_rank,
            self.num_heads * self.q_head_dim, 
            # 最终还需要做切分（split），一部分是 nope，一部分需要应用 rope
            bias=False,
        )

        # 同理对于 kv 也是一样的
        self.kv_down_proj = nn.Linear(
            self.hidden_size,
            self.kv_lora_rank + self.qk_rope_head_dim,
            bias=config.attention_bias,
        )
        self.kv_down_layernorm = DeepseekV2RMSNorm(self.kv_lora_rank)
        self.kv_up_proj = nn.Linear(
            self.kv_lora_rank,
            self.num_heads * (
                self.q_head_dim - self.qk_rope_head_dim + self.v_head_dim
            ), # 其中 self.q_head_dim - self.qk_rope_head_dim 是 nope 部分
            bias=False,
        )

        # 对应公式 47 行
        self.o_proj = nn.Linear(
            self.num_heads * self.v_head_dim,
            self.hidden_size,
            bias=config.attention_bias,
        )

        # 初始化 rope 的参数
        self.rotary_emb = DeepseekV2RotaryEmbedding(
            self.qk_rope_head_dim,
            self.max_postion_embeddings,
            self.rope_theta,
        )